## CRM-KBOX PROMED REENCODER: CLIENTS

In [ ]:
#Installs
!pip install xlrd
!pip install openpyxl

In [ ]:
#Imports
import numpy as np
import pandas as pd 
import pycountry
import csv

from datetime import datetime
from contextlib import redirect_stdout #For the log files

In [ ]:
#file_name = "../input/promed/Ventas Regulares Corregido 2.xlsx"
#df = pd.read_excel(file_name, engine='openpyxl', sheet_name='9ba16c51a1fe738624fe8dd92edc9b4')
file_name = "../input/promed/PROMEDmaestroCLIENTES-WORK.xlsx"
df = pd.read_excel(file_name, sheet_name='TOTALES')

In [ ]:
df = df.iloc[:,:7].copy()
df.head()

In [ ]:
#df[:1000].to_csv('shorts', index = False)
#df[:1000].to_excel("shorts.xlsx", sheet_name='Sheet_name_1')  
#file_name = "./shorts.xlsx"
#df1 = pd.read_excel(file_name)
#df_s = df[:1000].copy()

### 1) Encoder Clientes

In [ ]:
##0 - Saving cedula - cli_concat relationships
def cedula_cli_save(table):
    
    ced_un = table['CEDULA'].unique()
    cedula_dict = {}
    
    try:

        for key in ced_un:
            cli_list = []
            key_list = table[(table['CEDULA'] == key)]

            if len(key_list) > 1:
                          cedula_dict[key] = list(key_list['CLI-CONCAT'])

            else: continue
        
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished cedula - cli saving')
        return cedula_dict
    
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass


#df = df[df.duplicated(['CEDULA'], keep=False)]
#ced_un = df['CEDULA'].unique()
#cedula_dict = {}
#
#for key in ced_un:
#    cli_list = []
#    for i in range(len(df)): 
#        if key == df.iloc[i,5]:
#            cli_list.append(df.iloc[i,5])
#        else:
#            continue
#    cedula_dict[key] = cli_list 

In [ ]:
saved_dict = cedula_cli_save(df)

In [ ]:
saved_file = open("saved.csv", "w")

writer = csv.writer(saved_file)
for key, value in saved_dict.items():
    writer.writerow([key, value])

saved_file.close()

In [ ]:
len(df['CEDULA'].unique())
df = df.drop_duplicates('CEDULA')

df.head()

In [ ]:
#1 - Countries 
#def countryReencoder(table):
#    ##Existing country codes
#    num2countryDict = { 1 : 'Panama', 12 : 'Costa Rica', 
#                       15 : 'El Salvador', 18 : 'Guatemala', 
#                       22 : 'Honduras', 24 : 'República Dominicana', 
#                       26 : 'Nicaragua', }
#    
#    try:
#        ##Create the ISO for the existing countries
#        coutry2IsoDict = {}
#        for country in pycountry.countries:
#            coutry2IsoDict[country.name] = country.alpha_3
#
#        ##Iso codes alpha 3
#        countries = table['No Cia'].map(num2countryDict).map(coutry2IsoDict).fillna(table['No Cia']).astype('string')
#        
#        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished country encoding')
#        return countries
#         
#    except:
#        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
#        pass

#coutry2IsoDict = {
#    'Panama':,
#    'Costa Rica':,
#    'El Salvador':,
#    'Guatemala':,
#    'Honduras':,
#    'República Dominicana':,
#    'Nicaragua:,
#}

#codes = [countries.get(country, 'Unknown code') for country in input_countries]
#coutry2Iso = (df_s.replace({'No Cia':num2countryDict})).replace({'No Cia':coutry2IsoDict})
#coutry2Iso

In [ ]:
df['PAÍS'].unique()

In [ ]:
#1 - Countries 
def countryReencoder(table):
    try:
        ## Special conditions
        conditions = [ table['PAÍS'].eq('PA') | table['PAÍS'].eq('001'),
                     table['PAÍS'].eq('CR'),
                     table['PAÍS'].eq(float('nan')), ]
        
        choices = ['PAN', 'CRI', 'XXX']
        
        table['countries'] = np.select(conditions, choices, default=table['PAÍS'])
        table['countries'] = table['countries'].fillna('XXX')
        table['countries'] =  table['countries'].astype('string')
        
        countries = table['countries'].to_numpy()
        table.drop(['countries',], axis=1, inplace=True)
        
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished country encoding')
        return countries
    
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass

In [ ]:
a = countryReencoder(df)

In [ ]:
a[:50]

In [ ]:
##2 - Client Typology
def clientTypeEncoder(clientType = 'XX'):
    
    typeDict = { 'Organismo Público Salud': '110', 'Organismo Públio No Salud': '120',
                 'Hospital Público': '130', 'Laboratorio Público': '140',
                 'Empresa Privada Salud': '210', 'Empresa Privada No Salud': '220',
                 'Hospital Privado': '230', 'Laboratorio Privado': '240',
                 'Consulta Médica Privada': '250', } ## tipus
    
    try:   
        if clientType in typeDict.keys():
            typeCode = typeDict[clientType]
            
            print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished client type encoding. Defined')
            return typeCode
        
        else:
            typeCode = 'XX' # For undefined types
            print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished client type encoding. Undefined')
            return typeCode
         
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass

In [ ]:
cl = 'Organismo Público Salud'
clientTypeEncoder(cl)

In [ ]:
##3 - World Hospitals Clusters 
def clientSizeEncoder(size = 'X'):
    
    ##Define hospital sizes
    sizeDict = { 'Group_0': '0', 'Group_1': '1', 'Group_2': '2',
                'Group_3': '3', 'Group_4': '4', 'Group_5': '5', }
    
    try:
        if size in sizeDict.keys():
            sizeCode = sizeDict[size]
            print(datetime.now().strftime("%d-%m-%Y%H:%M:%S"), 'Finished client size encoding. Defined')

            return sizeCode
        else:
            sizeCode = 'X' #For undefined hospitals
            
            print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished client size encoding. Undefined')
            return sizeCode
        
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass

In [ ]:
hos = 'Group_1'
clientSizeEncoder(hos)

In [ ]:
#4 - Num Secuencial Client
def clientNumReencoder(table):

    try:
        # Assigning numerical values and storing in another column
        num_client =  table['CEDULA'].astype('category')

        codes = num_client.cat.codes
        cats = num_client.cat.categories
        codesStr = 'C' + codes.astype('string').str.zfill(6)
        
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished client num encoding')
        return (codesStr, codes)
    
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass
        
#from sklearn.preprocessing import LabelEncoder
#le = LabelEncoder()
## Assigning numerical values and storing in another column
#df_s['No Cliente_Cat'] = le.fit_transform(df_s['No Cliente'])
#df_s['No Cliente_Cat']

#import uuid
#uniqueid = uuid.uuid1()

In [ ]:
clientNumReencoder(df)

In [ ]:
#def logFilename(inputs, filename):
#    f = open(filename, "a")
#    f.write("{0} -- {1}\n".format(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), inputs))
#    f.close()

In [ ]:
##5 - Saving cedula - cli relationships
def cedula_cli_save(table):
    
    ced_un = table['CEDULA'].unique()
    cedula_dict = {}
    
    try:

        for key in ced_un:
            cli_list = []
            key_list = table[(table['CEDULA'] == key)]

            if len(key_list) > 1:
                          cedula_dict[key] = list(key_list['CLI-CONCAT'])

            else: continue
        
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished cedula - cli saving')
        return cedula_dict
    
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass


#df = df[df.duplicated(['CEDULA'], keep=False)]
#ced_un = df['CEDULA'].unique()
#cedula_dict = {}
#
#for key in ced_un:
#    cli_list = []
#    for i in range(len(df)): 
#        if key == df.iloc[i,5]:
#            cli_list.append(df.iloc[i,5])
#        else:
#            continue
#    cedula_dict[key] = cli_list 

In [ ]:
def clientEncoder(table):
    try:
        with open('logfile.txt', 'a') as f:
            with redirect_stdout(f):

                table['cCountry'] = countryReencoder(table)
                cType = clientTypeEncoder()
                cSize = clientSizeEncoder()
                table['cNum'] = clientNumReencoder(table)[0]

                #'-'.join(strings)
                #client
                table['Codigo_KBOX_Cliente'] = table['cCountry'] + '-' + cType + '-' + cSize + '-' + table['cNum']
                table.drop(['cCountry', 'cNum'], axis=1, inplace=True)
                
                
                if (table['Codigo_KBOX_Cliente'].str.len() == 17).all():
                    print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished client complete sequence encoding. Defined')
                    return table['Codigo_KBOX_Cliente']
                
                else:
                    print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished client complete sequence encoding. Undefined')
                    return table['Codigo_KBOX_Cliente']
                    
    except: 
        with open('logfile.txt', 'a') as f:
            with redirect_stdout(f):
                print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass  

In [ ]:
clientEncoder(df)
df.to_excel("recoded.xlsx", sheet_name='Recoded', index = False)

In [ ]:
df

In [ ]:
len(df['NO_CLIENTE'].unique()), len(df['CEDULA'].unique()), df['CEDULA'].isnull()